# 디케이 투자에이전트 — 실행 노트북 (과제 제출용)

연세대학교 정보대학원 · AI핀테크 Agent 분석과 설계 · WEEK 06~10

이 노트북은 개인 투자자용 금융 분석 **AI Agent** 프로젝트가 아래 5개 핵심 요소를 **어떻게 구현했는지**를
설명하고, 프로젝트 코드를 직접 import 해 **로컬에서 실행 데모**로 보여준다.

| # | 요소 | 프로젝트 구현 위치 | 키 필요 |
|---|------|------------------|:---:|
| 1 | **Intent Classification** | `chat/intent.py` (TF-IDF char n-gram + LogisticRegression, 7분류 + 결정적 가드레일) | ✕ |
| 2 | **Prompt Routing** | `chat/build_prompt.py` (필수 블록·기준표 자동생성 + `tool_choice=auto`) | ✕ |
| 3 | **RAG 기반 문서 검색** | `rag/` (pdfplumber 청킹 · text-embedding-3-small · numpy 코사인) | ○ |
| 4 | **Tool Calling** | `chat/chat.py` · `chat/tools.py` (OpenAI function calling 루프) | ○ |
| 5 | **UI 구성 (선택)** | `frontend/` (좌 채팅 + 우 동적 패널 · `popupRouter.js`) | ✕ |

> **설계 철학(안전):** *판정·수치는 코드가 확정하고, LLM은 설명만 한다.* 매매 주문 API는 어디에도 없고(조회만),
> 위험 요청은 결정적 가드레일로 차단하며, 외부 콘텐츠(리포트·영상)는 출처를 귀속해 인용하고 면책을 붙인다.

> **프로젝트 범위:** 5주 로드맵(데이터 파이프라인 → 매크로 국면 엔진 → 종목 리포트 → LLM 챗봇 → 워치리스트)을
> 완성한 뒤 — 시황(매크로) 리포트 요약·챗 상담·'금일의 요약', **국면 이동 궤적(족적) 시각화**, 이동평균선 대순환,
> 일봉/주봉·기간 선택 차트, 회원제(인증·RBAC·질문 한도)·유저별 관심종목/대화기록, 헤드라인 통계까지 확장했다.
> **GCP Cloud Run + Cloud SQL** 로 프로덕션 배포돼 있고, GitHub Actions(WIF) CI/CD 로 main 자동배포된다.

---
## 0. 실행 환경 준비

**커널**: 프로젝트 가상환경(`.venv`)을 Jupyter 커널로 사용한다. 의존성은 프로젝트 루트에서 `uv sync`.

```bash
uv sync                                   # 의존성 설치(.venv)
.venv/bin/python -m ipykernel install --user --name stock-agent --display-name "Stock Agent (.venv)"
jupyter lab                               # 노트북 실행
```

키 없이도 1·2·5 섹션(순수 함수·ML)은 동작하고, 3·4 섹션(RAG 임베딩·chat)은 `OPENAI_API_KEY`가 필요하다
(아래 셀이 키 유무를 확인해 없으면 graceful 하게 넘어간다).

In [1]:
import sys, os

# 노트북이 notebooks/ 에서 열려도 프로젝트 루트를 sys.path 에 넣는다.
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))     # 프로젝트 루트의 .env (키는 여기서만 로드)

print("프로젝트 루트:", ROOT)
for k in ["OPENAI_API_KEY", "FRED_API_KEY", "KIS_APP_KEY"]:
    print(("[OK]" if os.environ.get(k, "").strip() else "[  ]"), k, "(등록됨)" if os.environ.get(k, "").strip() else "(미등록)")

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY", "").strip())

프로젝트 루트: /Users/koscom/Projects/Yonsei/stock-investment-agent
[OK] OPENAI_API_KEY (등록됨)
[OK] FRED_API_KEY (등록됨)
[OK] KIS_APP_KEY (등록됨)


---
## 1. Intent Classification (인텐트 분류)

사용자 질의를 **7개 의도**로 사전 분류한다. 목적은 두 가지 — ① **위험 요청 차단**(안전 게이트),
② 질의 성격 파악. 구현은 **2층 방어** 구조다.

- **1층 — 결정적 키워드 가드레일** (`chat/intent.py::guardrail_label`): 차단 4유형
  (① 단정 예측 요구, ② 내부정보 유도, ③ 과도한 위험 조장[빚·몰빵·전재산], ④ 시세조종)을 **정규식으로 즉시**
  `risk_guardrail` 로 귀속한다. **LLM을 호출하지 않고 코드가 결정**한다(안전 하한선).
- **2층 — ML 분류기** (`chat/intent.py::build_pipeline`): `TfidfVectorizer(analyzer="char_wb", ngram=(2,4))`
  + `LogisticRegression`. 한글을 **형태소 분석기 없이 문자 n-gram**으로 처리한다. 학습 데이터는
  `data/intent_dataset.tsv`(라벨별 60~80개), 재학습은 `uv run python -m chat.intent_train`.

**강화 작업 (이번 과제):** 최근 추가한 애널리스트 리포트 수집/검색 기능(`fetch_analyst_reports`·`search_report`)을
인텐트 계층이 인식하지 못했다 — 데이터셋에 리포트 관련 질의가 **0건**이라 "리포트 확보해줘"가 `stock_analysis`로
뭉뚱그려졌다. → **7번째 라벨 `analyst_report`를 신설**하고 데이터셋에 55개 질의를 보강해 재학습했다.

7분류: `macro_view` · `stock_analysis` · `portfolio_advice` · `watchlist_mgmt` · `general_qa` ·
**`analyst_report`(신설)** · `risk_guardrail`.

In [2]:
from chat.intent import classify, LABELS

print("인텐트 라벨(7):", ", ".join(LABELS))
print("-" * 66)

queries = [
    "지금 시장 국면하고 현금비중 어때?",        # macro_view
    "삼성전자 실적이랑 밸류에이션 분석해줘",       # stock_analysis
    "내 잔고에서 손실 난 종목 알려줘",           # portfolio_advice
    "관심종목 목록 보여줘",                     # watchlist_mgmt
    "PER이 뭐야?",                            # general_qa
    "삼성전자 애널리스트 리포트 확보해줘",         # analyst_report (강화)
    "이 종목 목표주가 리포트 찾아줘",            # analyst_report (강화)
    "빚내서 몰빵할까?",                        # risk_guardrail (가드레일)
]
for q in queries:
    print(f"{classify(q):16s} | {q}")

인텐트 라벨(7): macro_view, stock_analysis, portfolio_advice, watchlist_mgmt, general_qa, analyst_report, risk_guardrail
------------------------------------------------------------------
macro_view       | 지금 시장 국면하고 현금비중 어때?
stock_analysis   | 삼성전자 실적이랑 밸류에이션 분석해줘
portfolio_advice | 내 잔고에서 손실 난 종목 알려줘
watchlist_mgmt   | 관심종목 목록 보여줘
stock_analysis   | PER이 뭐야?
analyst_report   | 삼성전자 애널리스트 리포트 확보해줘
analyst_report   | 이 종목 목표주가 리포트 찾아줘
risk_guardrail   | 빚내서 몰빵할까?


아래는 **1층 결정적 가드레일**만 따로 본 것이다(ML 이전 단계). 위험 4유형 키워드가 있으면 `risk_guardrail`,
없으면 `None`(→ ML로 넘어감). 정당한 종목/조회 질의는 여기서 걸리지 않는다.

In [3]:
from chat.intent import guardrail_label

for q in [
    "이 종목 반드시 오르지?",      # ① 단정 예측
    "내부정보 있으면 알려줘",       # ② 내부정보
    "전재산 몰빵할까",            # ③ 과도한 위험
    "작전주 정보 좀 줘",          # ④ 시세조종
    "삼성전자 리포트 확보해줘",     # 정상 → None (ML이 analyst_report 로 분류)
]:
    print(f"{str(guardrail_label(q)):16s} | {q}")

risk_guardrail   | 이 종목 반드시 오르지?
risk_guardrail   | 내부정보 있으면 알려줘
risk_guardrail   | 전재산 몰빵할까
risk_guardrail   | 작전주 정보 좀 줘
None             | 삼성전자 리포트 확보해줘


**인텐트 → 우측 패널 결정적 라우팅** (최초 설계 반영). 인텐트 분류기가 **인자 없는 네비게이션 패널**
(시장국면/관심종목/잔고)을 직접 선택한다 — `chat/intent_panel.merge_intent_panel`이 인텐트 패널을 챗
응답 `popups` 맨 앞에 주입하고, 프론트가 `popups[0]`으로 우측 패널을 전환하므로 **인텐트가 권위적으로
패널을 결정**한다. ticker 가 필요한 종목리포트·args 가 필요한 편집은 인텐트가 인자를 못 뽑으므로 LLM
function calling 이 담당한다(편집·매매 패널은 인텐트에 매핑하지 않는다 — 오분류가 편집/주문 유발 0).

In [4]:
from chat.intent import classify
from chat.intent_panel import INTENT_PANEL, merge_intent_panel

print("인텐트 → 패널 매핑(결정적, 인자 없는 네비게이션 3종):")
for lbl, tool in INTENT_PANEL.items():
    print(f"  {lbl:16s} -> {tool}")
print("-" * 62)
for q in ["지금 시장 국면 어때?", "관심종목 목록 보여줘", "내 잔고 손익 알려줘", "삼성전자 분석해줘"]:
    intent = classify(q)
    popups = merge_intent_panel(intent, [])   # LLM 팝업 없이 인텐트만으로 패널 결정
    panel = popups[0]["name"] if popups else "(LLM 이 결정 — ticker/편집)"
    print(f"  {q:22s} -> intent={intent:16s} -> 패널: {panel}")

인텐트 → 패널 매핑(결정적, 인자 없는 네비게이션 3종):
  macro_view       -> show_macro_dashboard
  watchlist_mgmt   -> show_watchlist
  portfolio_advice -> show_balance
--------------------------------------------------------------
  지금 시장 국면 어때?           -> intent=macro_view       -> 패널: show_macro_dashboard
  관심종목 목록 보여줘            -> intent=watchlist_mgmt   -> 패널: show_watchlist
  내 잔고 손익 알려줘            -> intent=portfolio_advice -> 패널: show_balance
  삼성전자 분석해줘              -> intent=stock_analysis   -> 패널: (LLM 이 결정 — ticker/편집)


---
## 2. Prompt Routing (프롬프트 라우팅)

시스템 프롬프트는 `chat/build_prompt.py::build_prompt(judgement)`가 **매 호출 재조립**한다(국면 변경 자동 반영).
필수 블록(역할·판정출처고정·**기준표**·현재국면·설명지침·**팝업/콘텐츠 도구 규칙**·면책)으로 구성되며,
LLM은 `tool_choice="auto"`로 이 규칙에 따라 어떤 도구를 호출할지 스스로 라우팅한다.

핵심 설계는 **3중 일관성**이다 — 임계값 숫자를 프롬프트에 **하드코딩하지 않고** `macro.engine`의 상수
(`THRESHOLDS`·`REGIME_PARAMS`·`VIX_PANIC`)에서 **기준표를 자동 생성**한다. 상수 한 곳만 바꾸면 코드·프롬프트·
프론트가 동시에 반영된다.

`judgement`(국면 판정)는 결정적 엔진 `macro.engine.judge_regime(지표dict)`가 만든다 — 지표값만 주면 되므로
FRED 조회 없이도 데모할 수 있다.

In [5]:
from macro.engine import judge_regime
from chat.build_prompt import build_criteria_text, build_prompt

# 경기 악화(금리차 역전·HY 스프레드↑) + 심리 탐욕(VIX↑·공포탐욕 80) → '과열' 국면 예시
judgement = judge_regime({"yield_spread": -0.2, "hy_spread": 6.0, "vix": 40.0, "fear_greed": 80})
print("국면:", judgement["regime"],
      "| 권장 현금비중:", judgement["recommended_cash_ratio"], "%",
      "| 신뢰도:", judgement["confidence"])
print("=" * 66)
print("[판정 기준표 — macro.engine 상수에서 자동 생성(프롬프트 하드코딩 0)]\n")
print(build_criteria_text())

국면: 수축 | 권장 현금비중: 20 % | 신뢰도: medium
[판정 기준표 — macro.engine 상수에서 자동 생성(프롬프트 하드코딩 0)]

[국면 판정 기준 — 시스템이 이 규칙으로 판정함]
- 장단기 금리차: < 0이면 악화, > 0.5이면 양호, 0 ~ 0.5이면 중립
- HY 신용스프레드: > 5.0이면 악화, < 3.0이면 양호, 3.0 ~ 5.0이면 중립
- VIX 변동성: > 28이면 공포, < 14이면 탐욕, 14 ~ 28이면 중립
- 공포탐욕지수: < 25이면 공포, > 75이면 탐욕, 25 ~ 75이면 중립
- 신용·금리(경기축) 지표와 변동성·심리(심리축) 지표를 분리해 2축으로 판정
- 경보 플래그: VIX > 35이면 패닉 경보(표시용 플래그, 판정은 2축 로직이 결정)


In [6]:
import re

prompt = build_prompt(judgement)
print("시스템 프롬프트 길이:", len(prompt), "자\n")

# 라우팅 규칙 블록만 발췌: ⑥ 팝업 도구 · ⑧ 외부 콘텐츠 도구(리포트 두 채널 구분)
for marker, nxt in [("⑥", "⑦"), ("⑧", "⑨")]:
    m = re.search(re.escape(marker) + r".*?(?=" + re.escape(nxt) + r")", prompt, re.DOTALL)
    if m:
        print(m.group(0).strip()[:1000])
        print("-" * 66)

시스템 프롬프트 길이: 4413 자

⑥ [팝업 도구 사용 규칙]
- 사용자가 시장/국면/현금비중을 물으면 show_macro_dashboard 를 호출해 대시보드를 띄운다.
- 특정 종목 분석을 요청하면 show_stock_report(ticker 6자리)를 호출한다.
- 관심종목 목록을 원하면 show_watchlist 를 호출한다.
- 관심종목을 추가/제거하거나 매수/매도 목표가를 설정해 달라는 요청에는 manage_watchlist(action, ticker[, target_price(매수)][, sell_target_price(매도)])를 호출한다. 이는 "제안"일 뿐, 실제 변경은 사용자가 화면에서 확인(confirm)해야 반영된다 — 네가 직접 매매하거나 자동 실행하지 않는다.
- 사용자가 계좌 잔고·보유종목·평가액·수익/손실 현황을 물으면 show_balance 를 호출해 잔고 화면을 띄운다(파라미터 없음).
- show_balance/show_watchlist/show_stock_report 를 호출하면 서버가 그 화면의 요약 스냅샷(조회 시각 포함)을 tool 결과로 함께 제공할 수 있다. 그때는 **그 스냅샷에 적힌 숫자를 인용해 답해도 된다**(스냅샷에 없는 종목·수치는 지어내지 말고 "화면에 표시되지 않음"으로 다룬다). 스냅샷이 없으면 화면을 띄웠다고만 안내한다.
- 리밸런싱·분산·추가편입 상담은 국면 권장 현금비중·분산 원칙과 잔고 스냅샷에 근거해 **구체적 조정 방향과 후보를 제시**한다(actionable). 단 "반드시 오른다/수익 보장" 같은 단정은 금지하고 근거·위험·면책을 함께 밝힌다.
- 스냅샷은 조회 시각 기준이라 실시간과 다를 수 있음을 필요 시 환기한다. 판정·현금비중은 코드가 확정하며, 스냅샷에 없는 시세·재무 숫자를 네가 새로 만들지 않는다.
- 단순 용어 설명이나 위험 경고에는 도구를 호출하지 않고 텍스트로만 답한다.
----------------------------------------------------

---
## 3. RAG 기반 문서 검색

증권사 리포트 PDF를 근거로 답하기 위한 검색 파이프라인. 강의(W09)의 아이디어를 프로젝트 `rag/` 패키지로
재구현했다(FAISS 대신 **numpy 코사인**으로 의존성 절감).

1. **인입·청킹** `rag/ingest.py`: `pdfplumber`로 텍스트 추출 → 표 보존 청킹(`chunk_text`).
2. **임베딩** `rag/embed.py`: OpenAI `text-embedding-3-small`로 청크/쿼리 벡터화(배치).
3. **저장·검색** `rag/store.py`: 코사인 유사도 top-k(`ReportStore.search`), `.cache`에 영속.
   관리 API `POST /api/reports/reindex`, 챗 콘텐츠 툴 `search_report(query)`.

먼저 **청킹**은 키 없이 동작한다.

In [7]:
from rag import ingest

sample = ("삼성전자 3분기 실적은 시장 기대치를 상회했다. 메모리 업황 회복과 HBM 수요 확대가 실적을 견인했다. "
          "목표주가는 9만원으로 상향한다. 리스크 요인으로는 스마트폰 수요 둔화와 환율 변동이 있다. ") * 6
chunks = ingest.chunk_text(sample)
print("청크 수:", len(chunks))
print("첫 청크(140자):", chunks[0][:140])

청크 수: 1
첫 청크(140자): 삼성전자 3분기 실적은 시장 기대치를 상회했다. 메모리 업황 회복과 HBM 수요 확대가 실적을 견인했다. 목표주가는 9만원으로 상향한다. 리스크 요인으로는 스마트폰 수요 둔화와 환율 변동이 있다. 삼성전자 3분기 실적은 시장 기대치를 상회했다. 메모리


**임베딩 + 코사인 검색** (OpenAI 키 필요) — 자족 데모로 인라인 문서 3개를 벡터화해 쿼리와 가장 가까운
문서를 찾는다. 이것이 `ReportStore.search`가 내부적으로 하는 일이다.

In [8]:
import numpy as np

if HAS_OPENAI:
    from rag import embed
    docs = [
        "삼성전자 목표주가를 9만원으로 상향, 메모리 업황 회복과 HBM 수요 확대 기대",
        "현대차 3분기 영업이익 개선, 친환경차 판매 증가와 미국 시장 호조",
        "카카오 광고 매출 둔화로 목표주가 하향, 커머스 성장은 지속",
    ]
    M = embed.embed_texts(docs)                    # (3, 1536)
    q = embed.embed_query("삼성전자 목표주가 얼마야")   # (1536,)
    Mn = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
    qn = q / (np.linalg.norm(q) + 1e-9)
    sims = Mn @ qn                                 # 코사인 유사도
    for i in np.argsort(-sims):
        print(f"{sims[i]:.3f} | {docs[i]}")
else:
    print("OPENAI_API_KEY 없음 — 임베딩 데모 생략(청킹/코사인 구조는 위 셀 참고)")

0.737 | 삼성전자 목표주가를 9만원으로 상향, 메모리 업황 회복과 HBM 수요 확대 기대
0.353 | 카카오 광고 매출 둔화로 목표주가 하향, 커머스 성장은 지속
0.224 | 현대차 3분기 영업이익 개선, 친환경차 판매 증가와 미국 시장 호조


실제 `reports/` 폴더의 PDF를 인덱싱해 검색한다(`store.reindex` → `store.search_reports`).
샘플 리포트 `samsung_electronics_q3_2026.pdf`가 있으면 그 청크에서 검색 결과가 나온다.

In [9]:
if HAS_OPENAI:
    from rag import store
    info = store.reindex(folder="reports")          # reports/*.pdf → 청크·임베딩·인덱스
    print("인덱싱 결과:", info)
    hits = store.search_reports("삼성전자 목표주가와 리스크", top_k=3)
    for h in hits:
        print(f"\n[{h['source']}] score={h['score']:.3f}\n{h['text'][:180]}...")
    if not hits:
        print("(reports/ 에 PDF가 없으면 결과 없음 — PDF를 넣고 재실행하세요)")
else:
    print("OPENAI_API_KEY 없음 — RAG 검색 생략")

인덱싱 결과: {'reports': 1, 'chunks': 1, 'sources': ['samsung_electronics_q3_2026.pdf']}



[samsung_electronics_q3_2026.pdf] score=0.477
Samsung Electronics (005930) Q3 2026 Analyst Report
Investment opinion: BUY (maintained)
Target price: 95,000 KRW (upside 18 percent)
Current price: 80,500 KRW
Key thesis: Memory s...


---
## 4. Tool Calling (도구 호출)

챗봇은 OpenAI **function calling**으로 도구를 호출한다(`chat/chat.py::chat`). 도구는 두 종류다.

- **표시 툴(팝업)** `show_stock_report`·`show_macro_dashboard`·`show_watchlist`·`show_balance`·`manage_watchlist`:
  LLM은 *"무엇을 띄울지"*만 결정하고 **실데이터는 프론트가 직접 조회**한다(환각 차단). tool 결과로 `{ok:True}`만 되먹인다.
- **콘텐츠 툴** `summarize_youtube`·`search_report`·**`fetch_analyst_reports`**: **서버가 실행해 실제 텍스트를 되먹여**
  LLM이 요약한다. 예를 들어 "리포트 확보해줘"는 `fetch_analyst_reports(ticker)`가 네이버에서 수집·요약해 온다
  (출처 귀속·면책 유지).

도구 스키마는 키 없이 볼 수 있다.

In [10]:
from chat.tools import TOOLS, CONTENT_TOOLS, CHAT_MODEL

print("LLM 모델:", CHAT_MODEL, "| 도구 수:", len(TOOLS))
print("콘텐츠 툴(서버 실행·텍스트 되먹임):", ", ".join(sorted(CONTENT_TOOLS)))
print("-" * 66)
for t in TOOLS:
    fn = t["function"]
    kind = "콘텐츠" if fn["name"] in CONTENT_TOOLS else "표시  "
    params = ", ".join((fn["parameters"].get("properties") or {}).keys()) or "-"
    print(f"[{kind}] {fn['name']:22s} params: {params}")

LLM 모델: gpt-5.6-luna | 도구 수: 8
콘텐츠 툴(서버 실행·텍스트 되먹임): fetch_analyst_reports, search_report, summarize_youtube
------------------------------------------------------------------
[표시  ] show_macro_dashboard   params: highlight
[표시  ] show_stock_report      params: ticker, stock_name, focus
[표시  ] show_watchlist         params: sort_by
[표시  ] manage_watchlist       params: action, ticker, stock_name, target_price, sell_target_price
[표시  ] show_balance           params: -
[콘텐츠] summarize_youtube      params: video_url
[콘텐츠] search_report          params: query
[콘텐츠] fetch_analyst_reports  params: ticker


실제 한 턴을 실행한다(OpenAI 키 필요). "삼성전자 어때?"에 LLM이 `show_stock_report` 도구를 호출하고,
`{text, popups}`를 반환하는 것을 관측한다. `chat(user_query, judgement, session)` 시그니처.

In [11]:
if HAS_OPENAI:
    from chat.chat import chat
    from chat.session import Session

    session = Session()                    # 서버 세션(슬라이딩 윈도우 8)
    res = chat("삼성전자(005930) 지금 어때?", judgement, session)
    print("텍스트(앞 300자):\n", (res["text"] or "")[:300])
    print("\ntool_call(팝업 지시):")
    for p in res["popups"]:
        print("  -", p["name"], "|", p["args"])
else:
    print("OPENAI_API_KEY 없음 — chat 실행 생략(위 TOOLS 스키마 참고)")

텍스트(앞 300자):
 삼성전자(005930) 스냅샷 기준으로는 **실적 기대와 밸류에이션 부담이 함께 있는 종목**으로 보입니다.

- 현재가: **249,500원**, 전일 대비 **+2.25%**
- PER: **38.0배**  
  - PER은 주가가 이익의 몇 배로 거래되는지 보는 지표입니다.
- PBR: **3.9배**  
  - PBR은 주가가 순자산의 몇 배인지 나타냅니다.
- 52주 범위: **64,900~374,500원**
- 52주 위치: **59.6%**

### 리포트 의견
확보된 증권사 리포트에서는 다음과 같이 제시했습니다.

-

tool_call(팝업 지시):
  - show_macro_dashboard | {}
  - show_stock_report | {'focus': 'both', 'stock_name': '삼성전자', 'ticker': '005930'}


---
## 5. UI 구성 (선택)

프론트엔드(`frontend/`, React + Vite)는 **좌측 상시 채팅 + 우측 동적 패널** 2컬럼 구조다(팝업 모달 폐기).
LLM이 반환한 `popups`(tool_call name+args)를 `frontend/src/lib/popupRouter.js`의 **`POPUP_KIND`** 매핑으로
우측 패널 컴포넌트에 라우팅한다. 챗 응답은 **SSE 스트리밍**(진행 단계 체크리스트 + 토큰 타이핑)으로 렌더한다.

노트북에서는 UI를 실행하지 않고 매핑만 보여준다. 실제 화면은 아래로 띄운다.

```bash
# 터미널 1 — 백엔드
uv run uvicorn api.main:app --port 8000
# 터미널 2 — 프론트(Vite 개발 서버 → http://localhost:5173)
cd frontend && npm run dev
# 또는 도커 한 번에:  docker compose up --build
```

In [12]:
# tool_call name → 우측 패널 컴포넌트(frontend/src/lib/popupRouter.js 의 POPUP_KIND)
POPUP_KIND = {
    "show_stock_report":    "stock_report    (종목 종합리포트 + 이동평균선 대순환 카드)",
    "show_macro_dashboard": "macro_dashboard (시장 국면 게이지 · 판정근거 지표 카드)",
    "show_watchlist":       "watchlist       (관심종목 카드 로우 · 스파크라인)",
    "show_balance":         "balance         (잔고 네이비 히어로 카드)",
    "manage_watchlist":     "manage_watchlist(목표가 편집 확인 카드 — 사용자 확인 후 저장)",
}
print("LLM popups → 우측 동적 패널 라우팅:")
for k, v in POPUP_KIND.items():
    print(f"  {k:22s} → {v}")

LLM popups → 우측 동적 패널 라우팅:
  show_stock_report      → stock_report    (종목 종합리포트 + 이동평균선 대순환 카드)
  show_macro_dashboard   → macro_dashboard (시장 국면 게이지 · 판정근거 지표 카드)
  show_watchlist         → watchlist       (관심종목 카드 로우 · 스파크라인)
  show_balance           → balance         (잔고 네이비 히어로 카드)
  manage_watchlist       → manage_watchlist(목표가 편집 확인 카드 — 사용자 확인 후 저장)


---
## 보너스 — 결정적 엔진의 재사용: 국면 이동 궤적(족적)

"판정은 코드"의 강력한 예. 국면 판정 엔진 `judge_regime`은 **순수·결정적**(지표 dict → 국면)이라, 과거 월별
지표를 그대로 다시 먹이면 그 달의 국면을 **재현**할 수 있다. `macro/regime_history.build_trajectory`가 이를
이용해 경기×심리 매트릭스 위 **이동 궤적**을 만든다 — 라이브 판정과 **동일 함수**를 쓰므로 임계값·현금비중
3중 일관성이 자동 유지된다(LLM 미개입·네트워크 0). 화면 밀도는 범위별로 표본화한다(1년=분기·2년=반기·3년=연).

In [13]:
from macro.regime_history import build_trajectory, trajectory_step

def _series(pairs):
    return [{"date": d, "value": v} for d, v in pairs]

# 과거 4개월 월별 지표 → 각 달의 국면을 엔진으로 재현(같은 judge_regime, LLM 0)
months = build_trajectory({
    "yield_spread": _series([("2024-01-01", 0.6), ("2024-02-01", -0.2), ("2024-03-01", -0.3), ("2024-04-01", 0.7)]),
    "hy_spread":    _series([("2024-01-01", 2.5), ("2024-02-01", 6.0), ("2024-03-01", 6.5), ("2024-04-01", 2.0)]),
    "vix":          _series([("2024-01-01", 12.0), ("2024-02-01", 30.0), ("2024-03-01", 40.0), ("2024-04-01", 11.0)]),
    "fear_greed":   _series([("2024-01-01", 80.0), ("2024-02-01", 20.0), ("2024-03-01", 15.0), ("2024-04-01", 82.0)]),
})
print("월별 국면 재현(결정적 엔진, LLM 0):")
for p in months:
    print(f"  {p['date'][:7]} | 경기 {p['cycle_score']:+d} · 심리 {p['sentiment_score']:+d} -> {p['regime']}  (권장 현금 {p['recommended_cash_ratio']}%)")

print("\n범위별 표본 간격:", "1년 ->", trajectory_step(12)[1], "· 2년 ->", trajectory_step(24)[1], "· 3년 ->", trajectory_step(36)[1])

월별 국면 재현(결정적 엔진, LLM 0):
  2024-01 | 경기 +2 · 심리 +2 -> 확장  (권장 현금 60%)
  2024-02 | 경기 -2 · 심리 -2 -> 수축  (권장 현금 20%)
  2024-03 | 경기 -2 · 심리 -2 -> 수축  (권장 현금 20%)
  2024-04 | 경기 +2 · 심리 +2 -> 확장  (권장 현금 60%)

범위별 표본 간격: 1년 -> quarterly · 2년 -> semiannual · 3년 -> annual


---
## 6. 결론 — 다섯 요소가 어떻게 결합되나

한 번의 사용자 질의는 다음 파이프라인을 지난다:

```
사용자 질의
  → [1] Intent 분류 (7분류 + 결정적 위험 가드레일)        ← 위험이면 즉시 차단(LLM 미호출)
         └─ 네비게이션 인텐트(macro/watchlist/portfolio) → 우측 패널 결정적 라우팅(popups 앞 주입)
  → [2] Prompt Routing (build_prompt: 국면·기준표·도구규칙 주입 + tool_choice=auto)
  → [4] Tool Calling (LLM이 표시 툴/콘텐츠 툴 선택)
         ├─ 표시 툴  → 프론트가 실데이터 조회 → [5] 우측 패널 렌더
         └─ 콘텐츠 툴 → 서버 실행(예: [3] RAG search_report / 네이버 fetch_analyst_reports) → LLM 요약
  → 최종 답변(면책 포함) + 팝업 지시
```

**안전 불변식** — ① 매매 주문 API 없음(조회만), ② 국면·상태 **판정은 코드**(결정적 엔진)·LLM은 설명만,
③ 위험 요청은 결정적 가드레일로 차단, ④ 외부 콘텐츠(리포트·영상)는 **출처 귀속 + 면책**, ⑤ 임계값은
상수 단일 출처(3중 일관성).

**로드맵 이후 확장:** 시황(매크로) 리포트 요약·챗 상담·'금일의 요약'(최근 5개 종합·중복제거 10줄), **국면 이동
궤적 시각화**(위 보너스 — `judge_regime` 재현), 이동평균선 대순환·일봉/주봉·기간 선택 차트, 회원제(인증·RBAC·
질문 한도)·유저별 관심종목/대화기록, 헤드라인 통계까지 더해졌다. 신규 판정 로직도 모두 **코드가 결정하고
LLM은 설명/요약만** 하는 원칙을 지킨다.